# Pydantic Models & Validation Errors

**Goal:** define a request model that rejects bad input with a useful error message.

In this notebook you will build a book-checkout request model, step by step:

1. what a Pydantic model is and how `ValidationError` reports failures
2. field constraints that give you validation for free
3. ISBN-10 / ISBN-13 format validation with real checksum math
4. field validators that normalize input and craft error messages
5. due-date rules, including cross-field validation
6. the complete model: accepting good input, rejecting bad input

Each section pairs an explanation with a runnable code cell.

## 1 · Why validate at the boundary?

A request model is the contract at the edge of your system: before any business
logic runs, Pydantic checks that incoming data is well-formed and rejects it
with a `ValidationError` when it is not. The caller of an API is not
trustworthy — data may be malformed, incomplete, or plain wrong.

Validating at the boundary means:

- **fail early** — bad input never reaches the code that writes data or charges
  a user;
- **fail clearly** — the error is structured: you always know *which* field
  failed and *why*;
- **pass cleanly** — good input arrives already typed and normalized, so
  handlers can trust it.

A `ValidationError` is not a bug to hide. It is the intended, normal way a
model says "no".

In [1]:
from pydantic import BaseModel, ValidationError


class MemberRequest(BaseModel):
    """The smallest useful request model: two required fields."""

    name: str
    email: str


# Good input: the model is built and validated in one step.
good = MemberRequest.model_validate({"name": "Ada", "email": "ada@example.com"})
print("Accepted:", good)

# Bad input: rejected with a ValidationError that explains exactly what broke.
bad = {"name": "Ada"}  # 'email' is missing
try:
    MemberRequest.model_validate(bad)
except ValidationError as err:
    print("Rejected the request. The error report says:")
    for error in err.errors():
        print("  loc    :", error["loc"])
        print("  msg    :", error["msg"])
        print("  type   :", error["type"])

Accepted: name='Ada' email='ada@example.com'
Rejected the request. The error report says:
  loc    : ('email',)
  msg    : Field required
  type   : missing


## 2 · Field constraints give you free validation

Before reaching for custom code, declare the *shape* of your data with types and
constraints. Pydantic turns declarations like `min_length`, `gt` (greater than)
and defaults into automatic validation with consistent, built-in error messages.

Types do most of the work: Pydantic coerces and checks every field before your
custom validators run. Constraints from `Field()` are applied *after* type
coercion, so `"300"` can be accepted as an `int` and then checked against
`gt=0`.

Missing required fields produce a `missing` error; constraint violations produce
descriptive errors naming the limit that was crossed.

One caveat worth teaching: by default Pydantic *ignores* unknown keys. That is
forgiving, but it can hide typos (`"isbn"` written as `"isbn_number"`). The
final model in section 6 switches this off with `extra="forbid"` so unknown keys
become errors.

In [1]:
from pydantic import BaseModel, Field, ValidationError


class BookDraft(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    pages: int = Field(gt=0, le=10000)
    is_reference: bool = False


cases = [
    {"title": "", "pages": 300},  # title must have at least 1 character
    {"title": "Cooking", "pages": 0},  # pages must be greater than 0
    {"title": "Cooking", "pages": 300, "isbn": "0-306-40615-2"},  # unknown key
]

for payload in cases:
    try:
        model = BookDraft.model_validate(payload)
        print("Accepted:", model.title, "| unknown keys ignored by default")
    except ValidationError as err:
        print("Rejected:", payload)
        for error in err.errors():
            print(f"   {error['loc'][0]}: {error['msg']}  [{error['type']}]")

ModuleNotFoundError: No module named 'pydantic'

## 3 · ISBN format: the checksum math

An ISBN is not just "some digits" — both formats carry a **check digit** that
makes accidental typos detectable. Before asking Pydantic to validate an ISBN,
we implement the checksum rules as plain functions so they can be tested on
their own:

- **ISBN-10** — 10 characters; the last one may be `X` (value 10). The weighted
  sum `digit × (11 − position)` must be divisible by 11.
- **ISBN-13** — 13 digits starting with `978` or `979`. Digits alternate weights
  1 and 3; the total must be divisible by 10.

The checksum is what makes format validation *meaningful*: changing a single
digit will almost always break it, while a correctly written ISBN always passes.
That is the difference between "looks like digits" and "is a valid ISBN".

**Teaching point:** write these checks as pure functions first. They are easy to
unit-test in isolation, and you can reuse them in a Pydantic validator, a
standalone script, or a database-side check.

We also normalize the input first. ISBNs are commonly written with hyphens and
spaces (`978-0-306-40615-7`), but the checksum math runs on the bare digits —
so strip the decoration, then validate.

In [2]:
def normalize_isbn(raw: str) -> str:
    """Strip hyphens, spaces and dots before checking."""
    return raw.replace("-", "").replace(" ", "").replace(".", "")


def is_valid_isbn_10(isbn: str) -> bool:
    """ISBN-10: 10 characters, check digit may be 'X' (worth 10).

    Sum of digit * (11 - position) must be divisible by 11.
    """
    if len(isbn) != 10:
        return False
    total = 0
    for i, char in enumerate(isbn, start=1):
        if i == 10:
            if char not in "0123456789Xx":
                return False
            value = 10 if char in "Xx" else int(char)
        elif not char.isdigit():
            return False
        else:
            value = int(char)
        total += value * (11 - i)
    return total % 11 == 0


def is_valid_isbn_13(isbn: str) -> bool:
    """ISBN-13: 13 digits, starts with 978 or 979, weights 1,3,1,3...
    Total must be divisible by 10.
    """
    if len(isbn) != 13 or not isbn.isdigit():
        return False
    if not isbn.startswith(("978", "979")):
        return False
    total = sum(int(d) * (1 if i % 2 == 0 else 3) for i, d in enumerate(isbn))
    return total % 10 == 0


# Known-good and known-bad examples, with what each one should do.
tests = [
    ("978-0-306-40615-7", "ISBN-13, valid", True),
    ("0-306-40615-2", "ISBN-10, valid", True),
    ("0-975-22980-X", "ISBN-10 with an X check digit, valid", True),
    ("978-0-306-40615-8", "ISBN-13, bad checksum", False),
    ("0-306-40615-1", "ISBN-10, bad checksum", False),
    ("97803064061", "only 11 characters", False),
]

for raw, what, expected in tests:
    digits = normalize_isbn(raw)
    valid = (len(digits) == 10 and is_valid_isbn_10(digits)) or (
        len(digits) == 13 and is_valid_isbn_13(digits)
    )
    status = "OK" if valid == expected else "MISMATCH"
    print(f"{raw:<20} {what:<38} valid={valid!s:<5} expected={expected!s:<5} {status}")

978-0-306-40615-7    ISBN-13, valid                         valid=True  expected=True  OK
0-306-40615-2        ISBN-10, valid                         valid=True  expected=True  OK
0-975-22980-X        ISBN-10 with an X check digit, valid   valid=True  expected=True  OK
978-0-306-40615-8    ISBN-13, bad checksum                  valid=False expected=False OK
0-306-40615-1        ISBN-10, bad checksum                  valid=False expected=False OK
97803064061          only 11 characters                     valid=False expected=False OK


## 4 · Field validators: turning rules into model behaviour

Pure functions are useful, but validation only protects you once it is wired into
the model. A `@field_validator` runs for one specific field *after* type
coercion, and its **return value replaces the field's value** — which is how we
normalize the ISBN and store the canonical form.

A validator communicates failure by raising `ValueError`; Pydantic catches it and
turns it into the field's error message. This is where you craft the *useful*
part of the error: say what was wrong, what was expected, and — when possible —
how to fix it.

Three practical tips:

- **Normalize and return the value** from the validator, so downstream code never
  sees hyphens or stray spaces.
- **Make the message specific**: "checksum invalid" and "wrong number of
  characters" are different problems with different fixes.
- **Keep validators focused on one field**; reach for a `@model_validator` only
  when a rule needs several fields at once (next section).

In [4]:
from pydantic import BaseModel, ValidationError, field_validator

# normalize_isbn / is_valid_isbn_10 / is_valid_isbn_13 come from the cell above.


class BookRequest(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    isbn: str

    @field_validator("isbn")
    @classmethod
    def _check_isbn(cls, value: str) -> str:
        digits = normalize_isbn(value)
        if len(digits) == 10 and is_valid_isbn_10(digits):
            return digits
        if len(digits) == 13 and is_valid_isbn_13(digits):
            return digits
        raise ValueError(
            f"'{value}' is not a valid ISBN: expected 10 or 13 characters "
            "(hyphens and spaces are allowed) with a correct checksum."
        )


cases = [
    {"title": "Clean Code", "isbn": "978-0-13-235088-4"},   # valid ISBN-13, hyphens
    {"title": "Domain-Driven Design", "isbn": "0-306-40615-2"},  # valid ISBN-10
    {"title": "Refactoring", "isbn": "978-0-13-235088-5"},   # wrong check digit
    {"title": "Patterns", "isbn": "12345"},                  # wrong length
]

for payload in cases:
    try:
        model = BookRequest.model_validate(payload)
        print(f"Accepted: '{model.title}' -> isbn normalized to '{model.isbn}'")
    except ValidationError as err:
        print(f"Rejected: isbn '{payload['isbn']}'")
        for error in err.errors():
            print(f"   {error['loc'][0]}: {error['msg']}")

Accepted: 'Clean Code' -> isbn normalized to '9780132350884'
Accepted: 'Domain-Driven Design' -> isbn normalized to '0306406152'
Rejected: isbn '978-0-13-235088-5'
   isbn: Value error, '978-0-13-235088-5' is not a valid ISBN: expected 10 or 13 characters (hyphens and spaces are allowed) with a correct checksum.
Rejected: isbn '12345'
   isbn: Value error, '12345' is not a valid ISBN: expected 10 or 13 characters (hyphens and spaces are allowed) with a correct checksum.


## 5 · Due-date rules and cross-field validation

Due dates carry business rules, not just type rules. We need three of them:

1. the due date must not be in the past,
2. it must stay within the maximum loan period (30 days),
3. it cannot be the checkout day itself.

The first two depend only on the due date; the third — and the window itself —
need the checkout date too.

Two validator kinds cover these cases:

- **`@field_validator`** — checks one field, but can read already-validated
  siblings from `info.data`. That is how `due_date` learns the `checkout_date`.
- **`@model_validator(mode="after")`** — runs once, after *all* fields are
  validated, so it can compare fields against each other. It is the right tool
  for the "not the same day" rule.

**Defense in depth:** the field validator already rejects past dates, yet the
model validator re-checks the ordering invariant. A little duplication is fine
when it keeps each rule's failure message precise and local to the rule that
caught it.

**One detail worth noticing:** errors from a model validator have an *empty*
`loc` — they concern the model as a whole rather than one field. When formatting
errors, treat an empty `loc` as "the model" so the message still reads clearly.

In [5]:
from datetime import date, timedelta

from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

MAX_LOAN_DAYS = 30


class LoanRequest(BaseModel):
    book_id: int = Field(gt=0)
    checkout_date: date = Field(default_factory=date.today)
    due_date: date

    @field_validator("due_date")
    @classmethod
    def _check_due_date(cls, value: date, info) -> date:
        today = info.data.get("checkout_date", date.today())
        if value < today:
            raise ValueError(
                f"due date {value.isoformat()} is before the checkout date "
                f"{today.isoformat()}"
            )
        if value > today + timedelta(days=MAX_LOAN_DAYS):
            raise ValueError(
                f"due date {value.isoformat()} is more than {MAX_LOAN_DAYS} days "
                f"after checkout ({today.isoformat()})"
            )
        return value

    @model_validator(mode="after")
    def _not_same_day(self) -> "LoanRequest":
        if self.due_date == self.checkout_date:
            raise ValueError(
                "due date cannot be the same day as the checkout date; "
                "the loan needs at least one full day"
            )
        return self


base = {"book_id": 42, "checkout_date": date(2026, 11, 15)}
cases = [
    {**base, "due_date": date(2026, 11, 30)},   # 15 days out -> OK
    {**base, "due_date": date(2026, 11, 10)},   # in the past -> reject
    {**base, "due_date": date(2026, 12, 31)},   # beyond 30 days -> reject
    {**base, "due_date": date(2026, 11, 15)},   # same day -> reject
]

for payload in cases:
    try:
        model = LoanRequest.model_validate(payload)
        print(f"Accepted: checkout {model.checkout_date}, due {model.due_date}")
    except ValidationError as err:
        print(f"Rejected: due {payload['due_date']}")
        for error in err.errors():
            where = error["loc"][0] if error["loc"] else "model"
            print(f"   {where}: {error['msg']}")

Accepted: checkout 2026-11-15, due 2026-11-30
Rejected: due 2026-11-10
   due_date: Value error, due date 2026-11-10 is before the checkout date 2026-11-15
Rejected: due 2026-12-31
   due_date: Value error, due date 2026-12-31 is more than 30 days after checkout (2026-11-15)
Rejected: due 2026-11-15
   model: Value error, due date cannot be the same day as the checkout date; the loan needs at least one full day


## 6 · The complete model: accept good input, reject bad input

Now we assemble everything into one request model — constraints, ISBN checksum
validation, due-date rules, and `extra="forbid"` so that a typo like
`isbn_number` is rejected instead of silently ignored. The `check()` helper
shows the accept/reject behaviour side by side.

Every technique from this notebook has a place in the final model:

- **Types + `Field` constraints** — cheap, automatic checks on shape and range.
- **`@field_validator`** — per-field rules that need custom logic or
  normalization (ISBN checksum, the due-date window).
- **`@model_validator`** — rules that compare multiple fields (same-day loan).
- **`extra="forbid"`** — turns unknown keys into errors, catching typos at the
  boundary.
- **Raising `ValueError` with clear text** — this is what makes the error message
  *useful* rather than just "invalid".

When the `ValidationError` reaches the caller, `loc` says which field, `msg`
says why, and `type` lets the API layer categorize the failure (missing field,
extra key, value error...). That is the whole point: the caller can fix their
request from the error alone.

In [6]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

# All pieces from the earlier sections, combined into one request model.


class BookCheckoutRequest(BaseModel):
    # Typos like "isbn_number" are rejected instead of silently ignored.
    model_config = ConfigDict(extra="forbid")

    title: str = Field(min_length=1, max_length=200)
    isbn: str
    borrower: str = Field(min_length=1)
    checkout_date: date = Field(default_factory=date.today)
    due_date: date

    @field_validator("isbn")
    @classmethod
    def _check_isbn(cls, value: str) -> str:
        digits = normalize_isbn(value)
        if len(digits) == 10 and is_valid_isbn_10(digits):
            return digits
        if len(digits) == 13 and is_valid_isbn_13(digits):
            return digits
        raise ValueError(
            f"'{value}' is not a valid ISBN: expected 10 or 13 characters "
            "(hyphens and spaces are allowed) with a correct checksum."
        )

    @field_validator("due_date")
    @classmethod
    def _check_due_date(cls, value: date, info) -> date:
        today = info.data.get("checkout_date", date.today())
        if value < today:
            raise ValueError(
                f"due date must not be in the past: got {value.isoformat()}, "
                f"but checkout date is {today.isoformat()}"
            )
        if value > today + timedelta(days=MAX_LOAN_DAYS):
            raise ValueError(
                f"due date {value.isoformat()} is more than {MAX_LOAN_DAYS} days "
                f"after checkout ({today.isoformat()})"
            )
        return value

    @model_validator(mode="after")
    def _loan_window(self) -> "BookCheckoutRequest":
        if self.due_date == self.checkout_date:
            raise ValueError(
                "due date cannot be the same as the checkout date; "
                "the loan needs at least one full day"
            )
        return self


def check(label: str, payload: dict) -> None:
    print(f"\n--- {label} ---")
    try:
        model = BookCheckoutRequest.model_validate(payload)
        print("OK  ->", model)
    except ValidationError as err:
        print("FAIL -> the request was rejected:")
        for error in err.errors():
            loc = ".".join(str(p) for p in error["loc"]) or "model"
            print(f"   * {loc}: {error['msg']}  [{error['type']}]")


base = {
    "title": "The Pragmatic Programmer",
    "isbn": "978-0-306-40615-7",
    "borrower": "ada@example.com",
    "checkout_date": date(2026, 11, 15),
    "due_date": date(2026, 11, 30),
}

check("Valid request (ISBN-13)", base)
check("Valid request (ISBN-10)", {**base, "isbn": "0-306-40615-2"})
check("Bad ISBN checksum", {**base, "isbn": "978-0-306-40615-8"})
check("Due date in the past", {**base, "due_date": date(2026, 11, 10)})
check("Due date beyond 30 days", {**base, "due_date": date(2026, 12, 31)})
check("Missing borrower", {k: v for k, v in base.items() if k != "borrower"})
check("Typo: unknown field", {**base, "isbn_number": "978-0-306-40615-7"})

print("\nSummary of what each rule protects:")
print("  types + Field      -> shape and range of every field")
print("  ISBN field validator -> the ISBN really is a valid ISBN-10/ISBN-13")
print("  due-date validator  -> no past dates, no loans beyond 30 days")
print("  model validator     -> a loan always spans at least one full day")
print("  extra='forbid'      -> typos and unknown keys cannot sneak through")


--- Valid request (ISBN-13) ---
OK  -> title='The Pragmatic Programmer' isbn='9780306406157' borrower='ada@example.com' checkout_date=datetime.date(2026, 11, 15) due_date=datetime.date(2026, 11, 30)

--- Valid request (ISBN-10) ---
OK  -> title='The Pragmatic Programmer' isbn='0306406152' borrower='ada@example.com' checkout_date=datetime.date(2026, 11, 15) due_date=datetime.date(2026, 11, 30)

--- Bad ISBN checksum ---
FAIL -> the request was rejected:
   * isbn: Value error, '978-0-306-40615-8' is not a valid ISBN: expected 10 or 13 characters (hyphens and spaces are allowed) with a correct checksum.  [value_error]

--- Due date in the past ---
FAIL -> the request was rejected:
   * due_date: Value error, due date must not be in the past: got 2026-11-10, but checkout date is 2026-11-15  [value_error]

--- Due date beyond 30 days ---
FAIL -> the request was rejected:
   * due_date: Value error, due date 2026-12-31 is more than 30 days after checkout (2026-11-15)  [value_error]

--- Mi